In [3]:
from manim import *

video_scene = " -v WARNING --disable_caching srt_Scene"
image_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s srt_Scene"

video_static_scene = " -v WARNING --disable_caching srt_static_Scene"
image_static_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s srt_static_Scene"


video_contraction_scene = " -v WARNING --disable_caching srt_contraction_Scene"
image_contraction_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s srt_contraction_Scene"

In [4]:
class Clock(Mobject):
	def __init__(self, clock_center = np.array([0, 0, 0]), clock_radius = 1, **kwargs):
		super().__init__(**kwargs)

		self.clock_center = clock_center
		self.clock_radius = clock_radius

		clock_circle_filled = Circle(radius = self.clock_radius, color = WHITE, fill_color = WHITE, fill_opacity = 0.9).move_to(self.clock_center)
		clock_circle = Circle(radius = self.clock_radius, color = BLACK).move_to(self.clock_center)
		self.add(clock_circle_filled, clock_circle)

		clock_tick_line_group = VGroup()
		for i in range(12):
			clock_tick_line = Line(start = clock_center+clock_radius*UP, end = clock_center+clock_radius*UP*0.8, color = BLACK).rotate(angle = -i * 2*PI / 12, about_point = clock_center)
			clock_tick_line_group.add(clock_tick_line)
		self.add(clock_tick_line_group)


	def get_time(self, time = 0, color = DARK_GRAY, xshift = 0):
		time_group = VGroup()

		time_pointer = Line(
			start = self.clock_center, end = self.clock_center+self.clock_radius*0.7*UP, color = color, stroke_width = 6
			).add_tip(tip_width = 0.0655, tip_length = 0.1).rotate(angle = -time * 2*PI / 60, about_point = self.clock_center)
		time_group.add(time_pointer)


		time_center_circle = Circle(radius = 0.025, color = GRAY, fill_color = GRAY).move_to(self.clock_center)
		time_group.add(time_center_circle)
		time_group.shift(xshift*RIGHT)
		return time_group



class Light(Mobject):
	def __init__(self, light_start = np.array([0, 0, 0]), direction = RIGHT, light_color = RED, **kwargs):
		super().__init__(**kwargs)

		self.light_start = light_start
		self.direction = direction
		self.light_color = light_color

		self.lamp = ImageMobject("../external_media/lamp.png").scale(0.2).move_to(light_start)
		self.lamp.shift((self.lamp.get_right()-self.lamp.get_center()-0.5)*LEFT)
		self.lamp_left = ImageMobject("../external_media/lamp_left.png").scale(0.2).move_to(light_start)
		self.lamp_left.shift((self.lamp_left.get_center()-self.lamp_left.get_left()-0.5)*RIGHT)
		if (self.direction[0] < 0):
			self.lamp = self.lamp_left
		self.add(self.lamp)

		# light start (left, right)
		self.lamp_right_start = self.lamp.get_right()
		self.lamp_left_start = self.lamp.get_left()
		


	def get_ray(self, velocity = 1, time = 0, ship_width = 5):
		upper_limit = ship_width/2 - 0.5
		# sine to the right
		if (self.direction[0] > 0):
			sine_graph = FunctionGraph(lambda x: 0, x_range = [0, min([velocity*time, upper_limit])], color = self.light_color, stroke_width = 3)
			sine_graph.move_to(self.lamp.get_right()+(sine_graph.get_center()-sine_graph.get_left())*RIGHT)
		# sine to the left
		else:
			sine_graph = FunctionGraph(lambda x: 0, x_range = [-min([velocity*time, upper_limit]), 0], color = self.light_color, stroke_width = 3)
			sine_graph.move_to(self.lamp.get_left()+(sine_graph.get_center()-sine_graph.get_left())*LEFT)

		return sine_graph

In [5]:
class SpaceShip(Mobject):
	def __init__(self, ship_center = np.array([0, 0, 0]), ship_width = 5, ship_height = 2, ship_velocity = 1, c = 2, **kwargs):
		super().__init__(**kwargs)

		self.ship_center = ship_center
		self.ship_width = ship_width
		self.ship_height = ship_height
		self.ship_velocity = ship_velocity
		self.c = c

		# spaceship
		self.spaceship_group = Group()
		# spaceship_rect_bg = Rectangle(width = ship_width, height = ship_height,).move_to(ship_center)
		spaceship_rect = Rectangle(width = ship_width, height = ship_height, stroke_color = BLACK, fill_color = WHITE, fill_opacity = 1).move_to(ship_center)
		self.spaceship_group.add(spaceship_rect)

		self.person = ImageMobject("../external_media/transparent_markus.png").scale(0.15).move_to(spaceship_rect.get_corner(DOWN))
		self.person.shift(self.person.height/2*UP)
		self.spaceship_group.add(self.person)

		# parameters
		self.clock_height = 0.75
		self.light_height = 1.65

		# left clock
		self.left_clock_center = spaceship_rect.get_corner(DOWN+LEFT)+self.clock_height*UP
		self.left_clock = Clock(clock_center = self.left_clock_center, clock_radius = 0.5)
		self.left_time = self.left_clock.get_time(0)
		self.spaceship_group.add(self.left_clock)

		# right clock
		self.right_clock_center = spaceship_rect.get_corner(DOWN+RIGHT)+self.clock_height*UP
		self.right_clock = Clock(clock_center = self.right_clock_center, clock_radius = 0.5)
		self.right_time = self.right_clock.get_time(0)
		self.spaceship_group.add(self.right_clock)

		# left lamp
		self.light_left = Light(light_start = spaceship_rect.get_corner(DOWN)+self.light_height*UP, direction = LEFT)
		self.light_ray_left = self.light_left.get_ray()
		self.spaceship_group.add(self.light_left)

		# right lamp
		self.light_right = Light(light_start = spaceship_rect.get_corner(DOWN)+self.light_height*UP)
		self.light_ray_right = self.light_right.get_ray()
		self.spaceship_group.add(self.light_right)
		self.add(self.spaceship_group, self.left_time, self.right_time, self.light_ray_left, self.light_ray_right)


	def get_right_time(self, time):
		fixed_right_time = self.right_clock.get_time(time, color = RED, xshift = time*self.ship_velocity/5)
		self.spaceship_group.add(fixed_right_time)
		self.add(fixed_right_time)
		return fixed_right_time

	def get_left_time(self, time):
		fixed_left_time = self.left_clock.get_time(time, color = RED, xshift = time*self.ship_velocity/10)
		self.spaceship_group.add(fixed_left_time)
		self.add(fixed_left_time)
		return fixed_left_time


	def set_right_time(self, time):
		self.right_time.become(self.right_clock.get_time(time))
	
	def set_left_time(self, time):
		self.left_time.become(self.left_clock.get_time(time))


	def get_right_light(self, time):
		self.light_ray_right.become(self.light_right.get_ray(velocity = self.c-self.ship_velocity, time = time, ship_width = self.ship_width))
		
	def get_left_light(self, time):
		self.light_ray_left.become(self.light_left.get_ray(velocity = self.c+self.ship_velocity, time = time, ship_width = self.ship_width))


	def move_ship(self, time = 0):
		self.left_time.shift(RIGHT * self.ship_velocity * time)
		self.right_time.shift(RIGHT * self.ship_velocity * time)
		self.spaceship_group.move_to(self.ship_center + RIGHT * self.ship_velocity * time)



class Earth(Mobject):
	def __init__(self, earth_center = np.array([0, 0, 0]), **kwargs):
		super().__init__(**kwargs)

		self.earth_center = earth_center
		self.earth = ImageMobject("../external_media/earth.png").scale(0.5).move_to(earth_center)
		self.add(self.earth)


	def move_earth(self, velocity = 1, time = 0):
		self.earth.move_to(self.earth_center + RIGHT * velocity * time)



class StarSky(Mobject):
	def __init__(self, list_of_centers = np.array([0, 0, 0]), star_center = np.array([0, 0, 0]), **kwargs):
		super().__init__(**kwargs)

		self.star_centers = list_of_centers
		self.all_star_center = star_center
		self.stars = Group()

		for star_center in self.star_centers:
			star = Star(6, outer_radius = 0.05, inner_radius = 0.02, color = GRAY, fill_color = GRAY).move_to(star_center)
			self.stars.add(star)

		self.stars.move_to(self.all_star_center)
		self.add(self.stars)


	def move_stars(self, velocity = 1, time = 0):
		self.stars.move_to(self.all_star_center + RIGHT * velocity * time)


class SpaceCoords(Mobject):
	def __init__(self, coord_center = np.array([0, 0, 0]), **kwargs):
		super().__init__(**kwargs)

		self.coord_center = coord_center

		# coordinate system
		x_range = np.array([-30, 30, 1])
		y_range = np.array([-20, 20, 1])
		x_length = 30
		y_length = 20
		self.npla = NumberPlane(
			x_range = x_range, y_range = y_range, x_length = x_length, y_length = y_length,
			x_axis_config = {"stroke_opacity": 0.05, "stroke_color": BLACK}, y_axis_config = {"stroke_opacity": 0.05, "stroke_color": BLACK}, background_line_style = {"stroke_opacity": 0.05, "stroke_color": BLACK}
			).move_to(self.coord_center)
		self.add(self.npla)


	def move_space(self, velocity = 1, time = 0):
		self.npla.move_to(self.coord_center + RIGHT * velocity * time)

In [6]:
%%manim -qh --fps 60 $video_scene


class srt_Scene(Scene):
	def construct(self):
		self.camera.background_color = WHITE

		CVC = Text('CVC', font_size = 12, weight = BOLD, color = WHITE, font = 'Latin Modern Sans').align_on_border(RIGHT + DOWN, buff = 0.2)
		self.add(CVC)

		np.random.seed(seed = 10)
		star_list = 2*(np.random.rand(100, 3) - 1) * np.array([11, 5, 0])
		# print(star_list)

		earth = Earth(earth_center = np.array([0, -3.5, 0]))
		stars = StarSky(list_of_centers = star_list)
		space = SpaceCoords(coord_center = np.array([0, -3.5, 0]))
		self.add(space, stars, earth)

		spaceship = SpaceShip(ship_center = np.array([-1, 1.5, 0]), ship_width = 9, ship_velocity = 0.5, c = 1.5)
		self.add(spaceship)

		def spaceship_updater(ship):
			t = time_tracker.get_value()
			ship.set_right_time(5*t)
			ship.set_left_time(10*t)
			ship.get_right_light(t)
			ship.get_left_light(t)
			ship.move_ship(time = t)


		time_tracker = ValueTracker(0)
		self.wait(2)

		spaceship.add_updater(spaceship_updater)
		self.play(time_tracker.animate.set_value(2), rate_func = linear, run_time = 4)
		fixed_time_left = spaceship.get_left_time(2*10)
		# self.add(fixed_time_left)
		self.play(time_tracker.animate.set_value(4), rate_func = linear, run_time = 4)
		fixed_time_right = spaceship.get_right_time(4*5)
		self.wait(3)

		# remove updater and prepare stretch
		# spaceship.remove_updater(spaceship_updater)
		# spaceship.add(spaceship.light_ray_right, spaceship.light_ray_left)
		# self.play(spaceship.animate.stretch(0.85, dim = 0), runt_time = 6)
		self.wait(3)

Manim Community v0.22.1

In [7]:
%%manim -qh --fps 60 $video_static_scene


class srt_static_Scene(Scene):
	def construct(self):
		self.camera.background_color = WHITE

		CVC = Text('CVC', font_size = 12, weight = BOLD, color = WHITE, font = 'Latin Modern Sans').align_on_border(RIGHT + DOWN, buff = 0.2)
		self.add(CVC)

		np.random.seed(seed = 10)
		star_list = 2*(np.random.rand(100, 3) - 1) * np.array([11, 5, 0])
		# print(star_list)

		earth = Earth(earth_center = np.array([4, -3.5, 0]))
		stars = StarSky(list_of_centers = star_list, star_center = np.array([4, 0, 0]))
		space = SpaceCoords(coord_center = np.array([0, 1.5, 0]))
		self.add(space, stars, earth)

		spaceship = SpaceShip(ship_center = np.array([0, 1.5, 0]), ship_width = 9, ship_velocity = 0, c = 1.5)
		self.add(spaceship)


		spaceship_next = SpaceShip(ship_center = np.array([-1, 1.5, 0]), ship_width = 9, ship_velocity = 0.5, c = 1.5)

		earth_next = Earth(earth_center = np.array([0, -3.5, 0]))
		stars_next = StarSky(list_of_centers = star_list)
		space_next = SpaceCoords(coord_center = np.array([4, -3.5, 0]))


		def spaceship_updater(ship):
			t = time_tracker.get_value()
			ship.set_right_time(7.5*t)
			ship.set_left_time(7.5*t)
			ship.get_right_light(t)
			ship.get_left_light(t)
			ship.move_ship(time = t)


		def earth_updater(earth):
			t = time_tracker.get_value()
			earth.move_earth(time = t, velocity = -1)

		def stars_updater(stars):
			t = time_tracker.get_value()
			stars.move_stars(time = t, velocity = -1)

		def space_updater(space):
			t = time_tracker.get_value()
			space.move_space(time = t, velocity = -1)


	
		time_tracker = ValueTracker(0)
		self.wait(2)

		spaceship.add_updater(spaceship_updater)
		earth.add_updater(earth_updater)
		stars.add_updater(stars_updater)
		# space.add_updater(space_updater)
		self.play(time_tracker.animate.set_value(4/1.5), rate_func = linear, run_time = 8/1.5)
		fixed_time_left = spaceship.get_left_time(20)
		fixed_time_right = spaceship.get_right_time(20)
		# self.add(fixed_time_left)
		self.play(time_tracker.animate.set_value(4), rate_func = linear, run_time = 8-8/1.5)
		self.wait(3)
	
		spaceship.remove_updater(spaceship_updater)
		earth.remove_updater(earth_updater)
		stars.remove_updater(stars_updater)
		# space.remove_updater(space_updater)

		self.play(FadeOut(space), FadeOut(spaceship), FadeIn(space_next), FadeIn(spaceship_next), run_time = 4)
		# self.play(FadeTransform(spaceship, spaceship_next), FadeTransform(earth, ear7th_next), run_time = 4)

Manim Community v0.22.1

In [77]:
class ClockRod(Mobject):
	def __init__(self, rod_center = np.array([0, 0, 0]), rod_width = 1.25, rod_height = 2, ship_velocity = 1, c = 2, **kwargs):
		super().__init__(**kwargs)

		self.rod_group = Group()
		self.rod_center = rod_center
		self.rod_width = rod_width
		self.rod_height = rod_height

		self.rod_top = rod_center + rod_height/2*UP
		self.rod_bottom = rod_center + rod_height/2*DOWN

		rod = Rectangle(width = rod_width, height = rod_height, stroke_color = BLACK, fill_color = WHITE, fill_opacity = 1, stroke_width = 2).move_to(rod_center)
		rod_line_up = Line(start = self.rod_top + rod_width/2*LEFT, end = self.rod_top + rod_width/2*RIGHT, color = BLACK, stroke_width = 6)
		rod_line_down = Line(start = self.rod_bottom + rod_width/2*LEFT, end = self.rod_bottom + rod_width/2*RIGHT, color = BLACK, stroke_width = 6)

		self.rod_lamp = ImageMobject("../external_media/lamp.png").rotate(-PI/2).scale(0.2).move_to(self.rod_top)#.shift(0.2*UP)
		self.rod_lamp_down = self.rod_lamp.get_bottom()

		self.rod_clock = Clock(clock_center = rod_center + (rod_height/2 + 0.6)*UP, clock_radius = 0.5)
		self.rod_time = self.rod_clock.get_time(0, color = RED)

		self.rod_group.add(rod, rod_line_up, rod_line_down)
		self.add(self.rod_group, self.rod_lamp, self.rod_clock, self.rod_time)


	def set_time(self, time):
		self.rod_time.become(self.rod_clock.get_time(time, color = RED))


	def get_vlight(self, t):
		vlight_dash_length = 0.025
		vlight_dashed_ratio = 0.5
		vlight_group = VGroup()
		t_eff = t % 10
		if t_eff <= 5:
			vlight_dot_pos = self.rod_top + t_eff/5 * self.rod_height*DOWN
			vlight = DashedLine(start = self.rod_top, end = vlight_dot_pos, color = PURE_RED, dash_length = vlight_dash_length, dashed_ratio = vlight_dashed_ratio)
			vlight_group.add(vlight)
			if t > 10:
				vlight_trans = DashedLine(start = self.rod_bottom, end = vlight_dot_pos, color = PURE_RED, stroke_opacity = 0.5, dash_length = vlight_dash_length, dashed_ratio = vlight_dashed_ratio)
				vlight_group.add(vlight_trans)
		else:
			vlight_dot_pos = self.rod_top + (10-t_eff)/5 * self.rod_height*DOWN
			vlight = DashedLine(start = self.rod_bottom, end = vlight_dot_pos, color = PURE_RED, dash_length = vlight_dash_length, dashed_ratio = vlight_dashed_ratio)
			vlight_trans = DashedLine(start = self.rod_top, end = vlight_dot_pos, color = PURE_RED, stroke_opacity = 0.5, dash_length = vlight_dash_length, dashed_ratio = vlight_dashed_ratio)
			vlight_group.add(vlight, vlight_trans)

		vlight_dot = Dot(radius = 0.05, color = PURE_RED).move_to(vlight_dot_pos)
		vlight_group.add(vlight_dot)
		return vlight_group


	def get_clight(self, t, target_length = 8):
		clight_dash_length = 0.025
		clight_dashed_ratio = 0.5
		clight_group = VGroup()
		if t <= 5:
			clight_dot_pos = self.rod_top + t/5 * self.rod_height*DOWN + t / 10 * target_length * RIGHT 
			clight = DashedLine(start = self.rod_top, end = clight_dot_pos, color = PURE_RED, dash_length = clight_dash_length, dashed_ratio = clight_dashed_ratio)
			clight_group.add(clight)
		else:
			clight1_dot_pos = self.rod_top + self.rod_height*DOWN + 5 / 10 * target_length * RIGHT 
			clight1 = DashedLine(start = self.rod_top, end = clight1_dot_pos, color = PURE_RED, dash_length = clight_dash_length, dashed_ratio = clight_dashed_ratio)
			clight_group.add(clight1)
			clight_dot_pos = self.rod_top + (10-t)/5 * self.rod_height*DOWN + t / 10 * target_length * RIGHT 
			clight = DashedLine(start = clight1_dot_pos, end = clight_dot_pos, color = PURE_RED, dash_length = clight_dash_length, dashed_ratio = clight_dashed_ratio)
			clight_group.add(clight)
		clight_dot = Dot(radius = 0.05, color = PURE_RED).move_to(clight_dot_pos)
		clight_group.add(clight_dot)
		return clight_group


	def move_rod(self, t, target_length = 8):
		self.rod_clock.move_to(self.rod_center + RIGHT * t / 10 * target_length + (self.rod_height/2 + 0.6)*UP)
		self.rod_time.shift(RIGHT * t / 10 * target_length)
		self.rod_group.move_to(self.rod_center + RIGHT * t / 10 * target_length)
		self.rod_lamp.move_to(self.rod_center + self.rod_height/2*UP + RIGHT * t / 10 * target_length)



class Ruler(Mobject):
	def __init__(self, ruler_start = np.array([-4, 0, 0]), ruler_end = np.array([4, 0, 0]), ruler_color = GRAY, **kwargs):
		super().__init__(**kwargs)

		self.ruler_start = ruler_start
		self.ruler_length = (ruler_end-ruler_start)[0]
		self.ruler_color = ruler_color

		ruler_main = Line(start = ruler_start, end = ruler_end, color = self.ruler_color)
		ruler = Group(ruler_main)

		# minor lines
		# for 

		# main lines
		for i in range(5):
			main_vertical = self.__get_vertical(self.ruler_length / 4 * i, 0.25)
			ruler.add(main_vertical)

		self.add(ruler)

	def __get_vertical(self, x_off, length):
		vertical = Line(start = self.ruler_start + x_off*RIGHT - length/2*UP, end = self.ruler_start + x_off*RIGHT + length/2*UP, color = self.ruler_color)
		return vertical


In [78]:
%%manim -qh --fps 60 $video_contraction_scene


class srt_contraction_Scene(Scene):
	def construct(self):
		self.camera.background_color = WHITE

		CVC = Text('CVC', font_size = 12, weight = BOLD, color = WHITE, font = 'Latin Modern Sans').align_on_border(RIGHT + DOWN, buff = 0.2)
		self.add(CVC)

		# moving rod
		rod_up = ClockRod(rod_center = np.array([-4, 1.5, 0]))

		# static rods
		rod_lower_left = ClockRod(rod_center = np.array([-4, -2, 0]))
		rod_lower_right = ClockRod(rod_center = np.array([4, -2, 0]))
		self.add(rod_up, rod_lower_left, rod_lower_right)

		# ruler
		ruler = Ruler(ruler_start = np.array([-4, -2.5, 0]), ruler_end = np.array([4, -2.5, 0]))
		self.add(ruler)

		# cross light
		rod_up_clight = rod_up.get_clight(0)
		rod_up.clight = rod_up_clight
		self.add(rod_up_clight)

		# vertical lights
		rod_lower_left_vlight = rod_lower_left.get_vlight(0)
		rod_lower_right_vlight = rod_lower_right.get_vlight(0)
		rod_lower_left.vlight = rod_lower_left_vlight
		rod_lower_right.vlight = rod_lower_right_vlight
		self.add(rod_lower_left_vlight, rod_lower_right_vlight)


		# moving rod updater
		def rod_updater_moving(rod):
			t = time.get_value()
			new_clight = rod.get_clight(t)
			rod.clight.become(new_clight)
			rod.set_time(3*t)
			rod.move_rod(t)


		# static rod updater
		def rod_updater_static(rod):
			t = time.get_value() / 4 * np.sqrt(4**2 + 8**2)
			new_vlight = rod.get_vlight(t)
			rod.vlight.become(new_vlight)
			rod.set_time(3*t)


		time = ValueTracker(0)
		self.wait(1.5)

		rod_up.add_updater(rod_updater_moving)
		rod_lower_left.add_updater(rod_updater_static)
		rod_lower_right.add_updater(rod_updater_static)

		self.play(time.animate.set_value(10), rate_func = linear, run_time = 5)
		self.wait(1.5) 

Manim Community v0.22.1

ffmpeg -f concat -i srt_spaceship_merge_list.txt -c copy srt_spaceship.mp4